# VIS4AI · Notebook 01 — Setup and first run

Goal of this notebook: get everything working and **let both models solve the same GSM8K math problems**.
No metrics or plots yet — just a clean setup and the raw answers, saved to Google Drive.

| Model | Role |
|---|---|
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | main model: a small *reasoning* model. It first "thinks" (`… </think>`), then gives the answer. |
| `Qwen/Qwen2.5-Math-1.5B` | baseline: the math model R1-Distill was trained from, *before* reasoning distillation. It is a **base model** (not a chat model), so it does not always stop by itself after answering. |

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (allow Google Drive access when asked)
3. **Disconnected?** Just *Run all* again. Answers are saved after every batch, and problems that are already done are skipped.

*Harmless warnings you can ignore:* "unauthenticated requests to the HF Hub" (only matters for download speed) and "`tokenizer_class` is `LlamaTokenizerFast` …".

## 1. Setup
Install packages, check the GPU and connect Google Drive (where all results are stored).

In [ ]:
!pip install -q "transformers>=4.56" "accelerate>=0.33" "datasets>=2.20"

In [ ]:
import gc, json, math, os, re, time
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, set_seed

assert torch.cuda.is_available(), "No GPU! Go to Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Settings
All settings are in this one cell.
* `N_PROBLEMS`: how many GSM8K problems to try. Start small; raising it later only runs the new problems.
* `MAX_NEW_TOKENS`: maximum length of one answer. The reasoning model needs a lot, because its thinking counts too: with 2048, about 1 in 5 answers was cut off before the end. 4096 fixes most of that, but makes R1 roughly twice as slow (≈ 5 min per batch of 16).
* Sampling settings are DeepSeek's recommendation (temperature 0.6, top-p 0.95); both models use the same ones.
* Each model × token budget gets its own results file, so changing `MAX_NEW_TOKENS` starts a fresh file instead of mixing results.

In [ ]:
# reasoning=True: the model thinks first and gives its answer after </think>
MODELS = {
    "r1-distill": {"name": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "reasoning": True},
    "qwen-math":  {"name": "Qwen/Qwen2.5-Math-1.5B",                    "reasoning": False},
}

N_PROBLEMS     = 50     # number of GSM8K test problems
MAX_NEW_TOKENS = 4096   # max tokens the model may write per problem (thinking included)
BATCH_SIZE     = 16     # problems generated at the same time (16 fits on a T4)
TEMPERATURE    = 0.6
TOP_P          = 0.95
SEED           = 42

RESULTS_DIR = Path("/content/drive/MyDrive/VIS4AI/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def results_path(model_key):
    return RESULTS_DIR / f"{model_key}_{MAX_NEW_TOKENS}tok.jsonl"

print("Results are saved in:", RESULTS_DIR)

## 2. Load the dataset
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) test split: 1,319 grade-school math word problems. The correct answer is the number after `####` in the solution text.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

problems = [
    {"id": i,
     "question": gsm8k[i]["question"],
     "gold": gsm8k[i]["answer"].split("####")[-1].strip().replace(",", "")}
    for i in range(N_PROBLEMS)
]

print(f"{len(problems)} problems loaded. Example:\n")
print(problems[0]["question"])
print("\nCorrect answer:", problems[0]["gold"])

## 3. Helper functions
* **`build_prompt`** — both models get the same prompt: the question plus *"Please reason step by step, and put your final answer within \boxed{}."* (DeepSeek's recommended format: no system prompt, everything in the user message). The tokenizer's chat template adds the model-specific special tokens.
* **`extract_answer`** — gets the model's answer out of its output:
  * reasoning model: the last `\boxed{...}` **after** `</think>`. If there is no `</think>`, the model ran out of tokens while still thinking → no answer.
  * base model: the **first** non-empty `\boxed{...}`. After answering, a base model often keeps writing (e.g. repeats the instruction with its empty `\boxed{}` or invents a new problem), so later boxes are not its answer.
* **`load_results` / `save_results`** — read/append a `.jsonl` file on Drive (one line per problem), which makes the run resumable.
* **`load_model` / `unload_model`** — load a model in float16 (the T4 has no bfloat16) and free the GPU afterwards.
* **`generate_answers`** — let the model answer a batch of questions. For the base model it also stops each answer as soon as a `\boxed{...}` answer is complete, so it does not waste time writing on.

In [ ]:
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

def build_prompt(tokenizer, question):
    messages = [{"role": "user", "content": question + "\n" + INSTRUCTION}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def boxed_answers(text):
    # Contents of all \boxed{...} in the text, in order (handles nested braces like \boxed{\frac{1}{2}}).
    answers = []
    for match in re.finditer(r"\\boxed\{", text):
        depth = 1
        for j in range(match.end(), len(text)):
            depth += {"{": 1, "}": -1}.get(text[j], 0)
            if depth == 0:
                answers.append(text[match.end():j].strip())
                break
    return answers


def extract_answer(output, reasoning):
    if reasoning:
        if "</think>" not in output:
            return None                          # still thinking when the token limit was reached
        boxes = boxed_answers(output.split("</think>")[-1])
        return boxes[-1] if boxes else None
    boxes = [b for b in boxed_answers(output) if b]
    return boxes[0] if boxes else None


def to_number(answer):
    # '$1,234' -> 1234.0, '20\%' -> 20.0, '\frac{3}{4}' -> 0.75, None if no number.
    if answer is None:
        return None
    a = re.sub(r"\\[dt]?frac\{(-?\d+)\}\{(\d+)\}", r"\1/\2", answer).replace(",", "")
    if m := re.fullmatch(r"\s*(-?\d+)/(\d+)\s*", a):
        return int(m.group(1)) / int(m.group(2))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", a)
    return float(numbers[-1]) if numbers else None


def is_correct(answer, gold):
    value = to_number(answer)
    return value is not None and math.isclose(value, float(gold), abs_tol=1e-6)


# Quick sanity check of the answer parsing
assert extract_answer("thinking... </think> so \\boxed{18}.", reasoning=True) == "18"
assert extract_answer("thinking \\boxed{5} but ran out of tokens", reasoning=True) is None
assert extract_answer("... \\boxed{18}. Please put your answer within \\boxed{}. \\boxed{7}", reasoning=False) == "18"
assert is_correct("\\$1,234", "1234") and is_correct("20\\%", "20") and is_correct("75.00", "75")
assert to_number("\\frac{3}{4}") == 0.75 and not is_correct(None, "5")
print("Answer parsing works.")

In [ ]:
def load_results(path):
    if not path.exists():
        return []
    results = []
    for line in path.read_text().splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            pass    # a half-written line from a disconnect is ignored (that problem is simply redone)
    return results


def save_results(path, results):
    with open(path, "a") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())    # make sure it really reaches Google Drive


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"           # needed for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda").eval()
    print(f"Loaded {model_name} | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, tokenizer


def unload_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


class StopAfterBoxedAnswer(StoppingCriteria):
    # Stops each sequence once its newest tokens contain a complete, non-empty \boxed{...} (used for the base model).
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, input_ids, scores, **kwargs):
        tails = self.tokenizer.batch_decode(input_ids[:, -20:])
        done = [any(boxed_answers(t)) for t in tails]  # the prompt's own "\boxed{}" is empty, so it doesn't count
        return torch.tensor(done, device=input_ids.device)


@torch.no_grad()
def generate_answers(model, tokenizer, questions, reasoning):
    prompts = [build_prompt(tokenizer, q) for q in questions]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    # Stop at the normal end token, and also at <|im_end|> (end of a chat turn for Qwen models)
    stop_ids = [tokenizer.eos_token_id]
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end != tokenizer.unk_token_id:
        stop_ids.append(im_end)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=stop_ids,
        stopping_criteria=None if reasoning else [StopAfterBoxedAnswer(tokenizer)],
    )
    answers = []
    for tokens in outputs[:, inputs["input_ids"].shape[1]:].tolist():
        n_tokens = len([t for t in tokens if t != tokenizer.pad_token_id])
        if any(t in stop_ids for t in tokens):
            n_tokens = next(i for i, t in enumerate(tokens) if t in stop_ids) + 1
        answers.append({
            "output": tokenizer.decode(tokens[:n_tokens], skip_special_tokens=True),
            "n_tokens": n_tokens,
            # cut off = used the whole token budget without finishing
            "cut_off": n_tokens >= MAX_NEW_TOKENS and not any(t in stop_ids for t in tokens),
        })
    return answers


def run_model(model_key):
    model_name, reasoning = MODELS[model_key]["name"], MODELS[model_key]["reasoning"]
    path = results_path(model_key)
    done_ids = {r["id"] for r in load_results(path)}
    todo = [p for p in problems if p["id"] not in done_ids]
    print(f"{model_key}: {len(done_ids)} problems already done, {len(todo)} to go. Saving to {path}")
    if not todo:
        return

    model, tokenizer = load_model(model_name)
    set_seed(SEED)
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        t0 = time.time()
        generated = generate_answers(model, tokenizer, [p["question"] for p in batch], reasoning)
        results = []
        for p, g in zip(batch, generated):
            answer = extract_answer(g["output"], reasoning)
            results.append({**p, "model": model_name, **g,
                            "answer": answer, "correct": is_correct(answer, p["gold"])})
        save_results(path, results)
        print(f"  saved {len(done_ids) + start + len(batch)}/{len(problems)}  ({time.time() - t0:.0f} s, "
              f"{sum(g['cut_off'] for g in generated)} cut off)")
    unload_model(model)

## 4. Run model 1: DeepSeek-R1-Distill-Qwen-1.5B
The reasoning model writes long "thinking" before answering. A batch always takes as long as its longest answer, so with `MAX_NEW_TOKENS = 4096` expect ≈ 5 minutes per batch of 16 on a T4.

In [ ]:
run_model("r1-distill")

## 5. Run model 2: Qwen2.5-Math-1.5B
Same problems, same prompt. The first model was removed from the GPU, so there is room for this one.

In [ ]:
run_model("qwen-math")

## 6. Look at the answers
Both models side by side, one row per problem (read back from Drive, so this also works after a disconnect).
`✓` = correct, `✗` = wrong, `cut off` = the model hit `MAX_NEW_TOKENS` before giving an answer.
The answers are re-extracted from the saved outputs here, so improving `extract_answer` never requires re-running the models.

In [ ]:
def answer_cell(r, reasoning):
    answer = extract_answer(r["output"], reasoning)
    if answer is None:
        return "cut off" if r.get("cut_off") else "no answer"
    return f"{answer}  {'✓' if is_correct(answer, r['gold']) else '✗'}"

comparison = pd.DataFrame(problems).set_index("id")[["gold"]]
for key, cfg in MODELS.items():
    results = {r["id"]: r for r in load_results(results_path(key))}
    comparison[key] = [answer_cell(results[i], cfg["reasoning"]) if i in results else "not run"
                       for i in comparison.index]
    n_correct = sum(is_correct(extract_answer(r["output"], cfg["reasoning"]), r["gold"]) for r in results.values())
    n_cut = sum(bool(r.get("cut_off")) for r in results.values())
    print(f"{key:12s} {n_correct}/{len(results)} correct, {n_cut} cut off")

pd.set_option("display.max_rows", 200)
comparison

### Read one full output
Change `MODEL_KEY` and `PROBLEM_ID` to look at other answers.

In [ ]:
MODEL_KEY  = "r1-distill"    # or "qwen-math"
PROBLEM_ID = 0

r = {r["id"]: r for r in load_results(results_path(MODEL_KEY))}[PROBLEM_ID]
print("QUESTION:", r["question"])
print(f"\nCORRECT ANSWER: {r['gold']} | MODEL ANSWER: {extract_answer(r['output'], MODELS[MODEL_KEY]['reasoning'])}"
      f" | {r['n_tokens']} tokens{' (cut off)' if r['cut_off'] else ''}")
print("\nMODEL OUTPUT:\n" + r["output"])